# Simulation Launcher

Control panel for the pension simulation pipeline. Define scenarios as declarative parameter sets, preview the exact commands they resolve to, launch them, and check outputs — no terminal needed.

**Workflow:** define `SCENARIOS` below → `preview()` → `launch(dry_run=True)` to sanity-check → `launch(dry_run=False)` to run → `inventory()` / `compare_exhaustion()` for a quick read. Full analysis of any run: open `analysis/results.ipynb` and set `RUN_TAG` to the scenario's tag.

**Key conventions** (see `scenarios.py`):
- Every scenario reuses the baseline **market seed (123)** unless overridden, so simulation column *n* is the same market history everywhere and scenarios are comparable path-by-path against the baseline and each other.
- `stage="asset"` scenarios (contribution policy, investment strategy, return assumptions) reuse the baseline deterministic outputs via `detal_run_tag` — they only rerun the ~2.5-minute asset stage.
- `stage="both"` scenarios (discount override, tier-file counterfactuals) rerun liabilities too (~10 min at 19-way parallel).
- Scenario settings are stored in every output pkl under `"scenario"` and in the run folder's `_manifest.csv`, so a run folder is always self-describing.

In [ ]:
from pathlib import Path
import sys

_here = Path.cwd().resolve()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import pandas as pd
import scenarios as sc
from scenarios import Scenario, contribution_grid, equity_grid

pd.set_option("display.max_colwidth", 200)
print(f"Project root : {sc.ROOT}")
print(f"Baseline run : {sc.BASELINE_RUN_TAG} (market seed {sc.BASELINE_SEED})")

## Current Run Inventory

Every run folder under `Results/Runs/`, with output counts and scenario provenance from its manifest.

In [ ]:
sc.inventory()

## Define Scenarios

A `Scenario` whose fields are left at defaults reproduces the baseline. Available levers:

| Lever | Fields | Stage |
|---|---|---|
| Contribution policy | `contrib_add` (pp of payroll), `policy_start` (year), `contrib_always` | asset |
| Investment strategy | `equity_share` (flat override), `derisk_to` + `derisk_years` (glidepath) | asset |
| Return assumptions | `stock_premium`, `stock_vol` | asset |
| Liability valuation | `discount_override` (e.g. AAA yield) | detal/both |
| Benefit rules | `tier_file` (e.g. no-reform counterfactual workbook) | detal/both |
| Sampling | `num_sim`, `seed`, `plans` | both |

Grid helpers: `contribution_grid(deltas, starts)` and `equity_grid(shares)` expand into scenario lists with systematic run tags.

In [ ]:
# --- examples: edit/extend, then run the preview and launch cells below ---

# Demo: small, fast sanity scenario (2 plans, 1000 sims, +2pp contributions)
SCENARIOS = [
    Scenario(name="demo: contrib +2pp now", run_tag="scn_demo_c2s0",
             plans="AZ06,NJ73", num_sim=1000,
             contrib_add=2.0, policy_start=0, contrib_always=True),
]

# Full contribution-policy grid (24 asset runs, ~1 hour):
# SCENARIOS = contribution_grid(deltas=(0.5, 1, 2, 3, 5, 10), starts=(0, 5, 10, 15))

# Investment-strategy comparison (4 asset runs, ~10 min):
# SCENARIOS = equity_grid(shares=(0.5, 0.3, 0.0)) + [
#     Scenario(name="derisk to 30% over 10y", run_tag="scn_glide30",
#              derisk_to=0.30, derisk_years=10),
# ]

# Market-value (AAA) liability revaluation (detal + asset, ~15 min):
# SCENARIOS = [
#     Scenario(name="AAA-discounted liabilities", run_tag="scn_aaa",
#              stage="both", discount_override=0.0469),
# ]

# No-reform counterfactual (needs the counterfactual workbook first):
# SCENARIOS = [
#     Scenario(name="no post-2007 reforms", run_tag="scn_noreform",
#              stage="both", tier_file="planchanges_main_2022_noreform.xlsx"),
# ]

print(f"{len(SCENARIOS)} scenario(s) defined")

## Preview

One row per scenario: which settings deviate from baseline, and the exact commands that will run.

In [ ]:
sc.preview(SCENARIOS)

## Launch

`dry_run=True` only prints commands. Flip to `False` to execute; each step streams to a log under `Results/Runs/[tag]/_logs/` and the returned table reports status and timing. `stop_on_error=True` aborts the queue on the first failure.

In [ ]:
status = sc.launch(SCENARIOS, dry_run=True)
status

## Check Outputs

Inventory of scenario runs plus a quick cross-scenario read: per-plan probability of asset exhaustion by the chosen horizon, one column per run tag (baseline first).

In [ ]:
scenario_tags = [s.run_tag for s in SCENARIOS]
display(sc.inventory([sc.BASELINE_RUN_TAG] + scenario_tags))

comparison = sc.compare_exhaustion([sc.BASELINE_RUN_TAG] + scenario_tags, horizon=35)
if comparison.empty:
    print("No scenario asset outputs yet - launch with dry_run=False first.")
else:
    display(comparison.round(3).sort_values(sc.BASELINE_RUN_TAG, ascending=False))

## Notes

- Scenario folders are ordinary run folders: `analysis/results.ipynb` analyzes any of them by setting `RUN_TAG` (asset-only scenario folders have no detAL pkls of their own; plan-level GASB metrics still load from the asset pkls).
- Contribution scenarios require detAL pkls that contain `EmployeeContributionRate` / `EmployerContributionRate` (saved by the fast runner since 2026-06-10). If a plan is skipped with a rate message, rerun the baseline detal stage.
- Deleting a scenario: delete its folder under `Results/Runs/`. The baseline is never touched by asset-only scenarios.